In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# Repository Clone
%cd /kaggle/working
!git clone https://github.com/LakshayBaijal/Computer-Vision_Assignments_Lakshay.git
%cd /kaggle/working/Computer-Vision_Assignments_Lakshay/Q1

/kaggle/working
Cloning into 'Computer-Vision_Assignments_Lakshay'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 30 (delta 1), reused 27 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 344.64 KiB | 3.45 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/kaggle/working/Computer-Vision_Assignments_Lakshay/Q1


In [35]:
from pathlib import Path

# Find Q1
q1_candidates = list(Path("/kaggle/working").rglob("Q1/train.py"))
assert q1_candidates, "Q1/train.py not found in /kaggle/working"
REPO = q1_candidates[0].parent

# Find dataset root (img + annots)
data_root = None
for d in Path("/kaggle/input").rglob("*"):
    if d.is_dir() and (d / "img").exists() and (d / "annots").exists():
        data_root = d
        break
assert data_root is not None, "Dataset with img/ and annots/ not found in /kaggle/input"

print("REPO:", REPO)
print("DATA_ROOT:", data_root)

REPO: /kaggle/working/Computer-Vision_Assignments_Lakshay/Q1
DATA_ROOT: /kaggle/input/datasets/lakshaybaijal/q1-dataset


In [36]:
%pip install -q einops matplotlib "numpy>=1.26,<2.2" opencv-python PyYAML tqdm shapely Pillow

Note: you may need to restart the kernel to use updated packages.


In [40]:
import yaml, time
from pathlib import Path

run_tag = time.strftime("%Y%m%d_%H%M%S")
cfg_in = REPO / "config" / "st.yaml"
cfg_out = Path(f"/kaggle/working/run5_obb_multibin60_{run_tag}.yaml")

with open(cfg_in, "r") as f:
    cfg = yaml.safe_load(f)

cfg["dataset_params"]["root_dir"] = str(data_root)

tp = cfg["train_params"]
tp["use_angle"] = True
tp["num_epochs"] = 10
tp["lr"] = 5e-4
tp["lr_steps"] = [7, 9]

# keep task_name relative to avoid path issues
tp["task_name"] = f"checkpoints_run5_obb_multibin60_{run_tag}"
tp["ckpt_name"] = f"run5_obb_multibin60_{run_tag}.pth"

# multi-bin settings (60° bins => 6 bins over 360°)
tp["angle_mode"] = "multibin"
tp["angle_pred_mode"] = "multibin"
tp["bin_size"] = 60
tp["angle_bin_size"] = 60
tp["num_bins"] = 6
tp["angle_num_bins"] = 6

with open(cfg_out, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Saved config:", cfg_out)
print("train_params:", tp)

Saved config: /kaggle/working/run5_obb_multibin60_20260405_145435.yaml
train_params: {'task_name': 'checkpoints_run5_obb_multibin60_20260405_145435', 'seed': 1111, 'infer_seed': 1122, 'acc_steps': 1, 'num_epochs': 10, 'lr_steps': [7, 9], 'lr': 0.0005, 'ckpt_name': 'run5_obb_multibin60_20260405_145435.pth', 'use_angle': True, 'angle_mode': 'multibin', 'angle_pred_mode': 'multibin', 'bin_size': 60, 'angle_bin_size': 60, 'num_bins': 6, 'angle_num_bins': 6}


In [41]:
import os, sys
os.chdir(str(REPO))
sys.path.insert(0, str(REPO))

from train import train

class Args:
    config_path = str(cfg_out)
    root_dir = str(data_root)

train(Args())
print("Run 5 training completed.")

{'dataset_params': {'root_dir': '/kaggle/input/datasets/lakshaybaijal/q1-dataset', 'num_classes': 2}, 'model_params': {'im_channels': 3, 'aspect_ratios': [0.5, 1, 2], 'scales': [128, 256, 512], 'min_im_size': 600, 'max_im_size': 1000, 'backbone_out_channels': 512, 'fc_inner_dim': 1024, 'rpn_bg_threshold': 0.3, 'rpn_fg_threshold': 0.7, 'rpn_nms_threshold': 0.7, 'rpn_train_prenms_topk': 12000, 'rpn_test_prenms_topk': 6000, 'rpn_train_topk': 2000, 'rpn_test_topk': 300, 'rpn_batch_size': 256, 'rpn_pos_fraction': 0.5, 'roi_iou_threshold': 0.5, 'roi_low_bg_iou': 0.0, 'roi_pool_size': 7, 'roi_nms_threshold': 0.3, 'roi_topk_detections': 100, 'roi_score_threshold': 0.05, 'roi_batch_size': 128, 'roi_pos_fraction': 0.25, 'bbox_reg_weights': [10.0, 10.0, 5.0, 5.0, 1.0]}, 'train_params': {'task_name': 'checkpoints_run5_obb_multibin60_20260405_145435', 'seed': 1111, 'infer_seed': 1122, 'acc_steps': 1, 'num_epochs': 10, 'lr_steps': [7, 9], 'lr': 0.0005, 'ckpt_name': 'run5_obb_multibin60_20260405_1454

100%|██████████| 181/181 [02:35<00:00,  1.16it/s]


Finished epoch 0
Checkpoint saved: checkpoints_run5_obb_multibin60_20260405_145435/tv_frcnn_r50fpn_run5_obb_multibin60_20260405_145435.pth
RPN Classification Loss : 0.1675 | RPN Localization Loss : 0.1081 | FRCNN Classification Loss : 0.9524 | FRCNN Localization Loss : 0.4091


100%|██████████| 181/181 [02:37<00:00,  1.15it/s]


Finished epoch 1
Checkpoint saved: checkpoints_run5_obb_multibin60_20260405_145435/tv_frcnn_r50fpn_run5_obb_multibin60_20260405_145435.pth
RPN Classification Loss : 0.0659 | RPN Localization Loss : 0.0900 | FRCNN Classification Loss : 0.8054 | FRCNN Localization Loss : 0.4125


...

Finished epoch 7
Checkpoint saved: checkpoints_run5_obb_multibin60_20260405_145435/tv_frcnn_r50fpn_run5_obb_multibin60_20260405_145435.pth
RPN Classification Loss : 0.0268 | RPN Localization Loss : 0.0596 | FRCNN Classification Loss : 0.5941 | FRCNN Localization Loss : 0.3364


100%|██████████| 181/181 [02:39<00:00,  1.14it/s]


Finished epoch 8
Checkpoint saved: checkpoints_run5_obb_multibin60_20260405_145435/tv_frcnn_r50fpn_run5_obb_multibin60_20260405_145435.pth
RPN Classification Loss : 0.0234 | RPN Localization Loss : 0.0563 | FRCNN Classification Loss : 0.5754 | FRCNN Localization Loss : 0.3307


100%|██████████| 181/181 [02:38<00:00,  1.15it/s]


Finished epoch 9
Checkpoint saved: checkpoints_run5_obb_multibin60_20260405_145435/tv_frcnn_r50fpn_run5_obb_multibin60_20260405_145435.pth
RPN Classification Loss : 0.0208 | RPN Localization Loss : 0.0533 | FRCNN Classification Loss : 0.5524 | FRCNN Localization Loss : 0.3214
Done Training...
Run 5 training completed.


In [46]:
import glob, os, json, shutil, time
from pathlib import Path

# expected directory
ckpt_dir = REPO / cfg["train_params"]["task_name"]
ckpts = sorted(glob.glob(str(ckpt_dir / "*.pth")))

# fallback search
if not ckpts:
    ckpts = sorted(glob.glob("/kaggle/working/**/*.pth", recursive=True), key=os.path.getmtime)

assert ckpts, "No checkpoint found after training"
latest = ckpts[-1]
print("Latest checkpoint:", latest)

tag = time.strftime("%Y%m%d_%H%M%S")
export_dir = Path(f"/kaggle/working/EXPORT_Q1_RUN5_OBB_MULTIBIN60_{tag}")
export_dir.mkdir(parents=True, exist_ok=True)

final_pth = export_dir / f"Q1_RUN5_OBB_MULTIBIN60_final_{tag}.pth"
shutil.copy2(latest, final_pth)

with open(export_dir / f"Q1_RUN5_OBB_MULTIBIN60_manifest_{tag}.json", "w") as f:
    json.dump({
        "run": "Q1_RUN5_OBB_MULTIBIN60",
        "latest_checkpoint": latest,
        "config": str(cfg_out),
        "data_root": str(data_root),
    }, f, indent=2)

zip_path = shutil.make_archive(f"/kaggle/working/Q1_RUN5_OBB_MULTIBIN60_EXPORT_{tag}", "zip", str(export_dir))
print("ZIP:", zip_path)

Latest checkpoint: /kaggle/working/Computer-Vision_Assignments_Lakshay/Q1/checkpoints_run5_obb_multibin60_20260405_145435/tv_frcnn_r50fpn_run5_obb_multibin60_20260405_145435.pth
ZIP: /kaggle/working/Q1_RUN5_OBB_MULTIBIN60_EXPORT_20260405_153852.zip


In [47]:
from IPython.display import display, FileLink, HTML
import os

zip_name = os.path.basename(zip_path)
display(FileLink(zip_path))
display(HTML(f'<a href="/files/{zip_name}" target="_blank">Download {zip_name}</a>'))
display(HTML(f'<script>window.open("/files/{zip_name}", "_blank");</script>'))
print("If popup blocked, click the link manually.")

/kaggle/working/Q1_RUN5_OBB_MULTIBIN60_EXPORT_20260405_153852.zip

If popup blocked, click the link manually.


In [44]:
from IPython.display import display, FileLink, HTML
import os

zip_name = os.path.basename(zip_path)
display(FileLink(zip_path))
display(HTML(f'<a href="/files/{zip_name}" target="_blank">Download {zip_name}</a>'))
display(HTML(f'<script>window.open("/files/{zip_name}", "_blank");</script>'))
print("If popup blocked, click the link above manually.")

/kaggle/working/Q1_RUN3_OBB_DIRECT_EXPORT_20260405_153458.zip

If popup blocked, click the link above manually.
